In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# download pre-trained model
pretrained_weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=pretrained_weights).to(device)

In [ ]:
# patch the model to provide also attention maps
def layer_forward(self, input: torch.Tensor):
    torch._assert(input.dim() == 3, f"Expected (batch_size, seq_length, hidden_dim) got {input.shape}")
    x = self.ln_1(input)
    x, att_map = self.self_attention(x, x, x, need_weights=True, average_attn_weights=False)
    att_maps.append(att_map)
    x = self.dropout(x)
    x = x + input
    y = self.ln_2(x)
    y = self.mlp(y)
    return x + y

for layer in model.encoder.layers:
    layer.forward = layer_forward.__get__(layer, type(layer))

In [ ]:
import kagglehub
path = kagglehub.dataset_download("ambityga/imagenet100")
print("Path to dataset files:", path)

In [ ]:
# download ImageNet Labels
!wget http://storage.googleapis.com/bit_models/ilsvrc2012_wordnet_lemmas.txt

In [ ]:
imagenet_labels = dict(enumerate(open('ilsvrc2012_wordnet_lemmas.txt')))
print(imagenet_labels)

In [ ]:
!ls -l $path/val.X/n01773549

In [ ]:
imagepath = path + '/val.X/' + 'n01773549' +'/' + 'ILSVRC2012_val_00008316.JPEG'

In [ ]:
import cv2
image = cv2.imread(imagepath)
from google.colab.patches import cv2_imshow
cv2_imshow(image)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

In [ ]:
size = (224, 224)

In [ ]:
blob = transform(cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2RGB),size)).unsqueeze(0).to(device)

In [ ]:
blob.shape

In [ ]:
with torch.no_grad():
    att_maps = []
    logits = model(blob)

In [ ]:
# Present probabilities of categories
probs = torch.sigmoid(logits)[0]
labels = torch.argsort(probs, dim=-1, descending=True)
top5 = labels[:5]
print("Prediction:")
for idx in top5:
    print(f'{probs[idx.item()]:.5f} : {imagenet_labels[idx.item()]}', end='')

In [ ]:
att_maps = torch.cat(att_maps, dim=0)

In [ ]:
att_maps.shape

In [ ]:
def draw_mask(img, mask):
    H, W = img.shape[:2]
    mask_resized = cv2.resize(mask, (W, H), interpolation=cv2.INTER_LINEAR)
    if img.ndim == 3:
        mask_resized = np.repeat(mask_resized[:, :, None], 3, axis=2)
    result = (img.astype(np.float32) * mask_resized).clip(0, 255).astype(np.uint8)
    return result

In [ ]:
base = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
num_layers, num_heads, N, _ = att_maps.shape
patch_size = int(math.sqrt(size[0]*size[1]//(N-1)))
att_size = (size[0]//patch_size, size[1]//patch_size)
plt.figure(figsize=(2 * num_heads, 2 * num_layers))
for t in range(num_layers):
    for i in range(num_heads):
        head = att_maps[t, i]
        mask = head[0, 1:].reshape(att_size[0], att_size[1])
        mask = mask.detach().cpu().numpy()
        disp = draw_mask(base, mask)
        plt.subplot(num_layers, num_heads, t * num_heads + i + 1)
        plt.imshow(disp, cmap='gray')
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# joint_attention pre layer
att = att_maps.mean(dim=1)
print(att.shape)

In [ ]:
# visualize averaged and normalizes heads
plt.figure(figsize=(num_layers,2))
for t in range(num_layers):
    head = att[t]
    mask = head[0, 1:].reshape(att_size[0], att_size[1])
    mask = mask.detach().cpu().numpy()
    disp = draw_mask(base, mask)
    plt.subplot(1, num_layers, t+1)
    plt.imshow(disp, cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()